In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"
model_id = "openai-community/gpt2"
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json: 100%|██████████| 665/665 [00:00<?, ?B/s] 
model.safetensors: 100%|██████████| 548M/548M [00:05<00:00, 96.3MB/s] 
generation_config.json: 100%|██████████| 124/124 [00:00<?, ?B/s] 
tokenizer_config.json: 100%|██████████| 26.0/26.0 [00:00<?, ?B/s]
vocab.json: 100%|██████████| 1.04M/1.04M [00:00<00:00, 18.1MB/s]
merges.txt: 100%|██████████| 456k/456k [00:00<00:00, 11.1MB/s]
tokenizer.json: 100%|██████████| 1.36M/1.36M [00:00<00:00, 16.1MB/s]


In [4]:
encodings = tokenizer("Hello, my dog is cute", return_tensors="pt").to(device)
encodings

{'input_ids': tensor([[15496,    11,   616,  3290,   318, 13779]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [18]:
import torch

def compute_nll(context: str, completion: str):
    context = tokenizer(context, return_tensors="pt").to(device)
    completion = tokenizer(completion, return_tensors="pt").to(device)
    input_ids = torch.cat([context["input_ids"], completion["input_ids"]], dim=-1)
    trg_len = completion["input_ids"].shape[-1]
    print(trg_len)
    target_ids = input_ids.clone()
    target_ids[:, :-trg_len] = -100
    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)
        loss = outputs.loss
    return loss.item() * trg_len

print(compute_nll("Hello, my dog is cute", "I like it very much"))
print(compute_nll("Hello, my dog is cute", "C++ is a programming language"))

5
24.15370464324951
6
29.999404907226562
